# System Dependencies
To get started with Unstructured.io, we need a few system-wide dependencies:

## Poppler (poppler-utils)
Handles PDF processing. It's a library that can extract text, images, and metadata from PDFs. Unstructured uses it to parse PDF documents and convert them into processable text.

## Tesseract (tesseract-ocr)
Optical Character Recognition (OCR) engine. When you have scanned documents, images with text, or PDFs that are essentially pictures, Tesseract reads the text from these images and converts it to machine-readable text.

## libmagic
File type detection library. It identifies what type of file you're dealing with (PDF, Word doc, image, etc.) by analyzing the file's content, not just the extension. This helps Unstructured choose the right processing method for each document.

In [ ]:
#%pip install -Uq "unstructured[all-docs]" 

Note: you may need to restart the kernel to use updated packages.


In [1]:
import json
from typing import List
import warnings
import os
import importlib
import torch
import tqdm as notebook_tqdm

# Unstructured for document parsing (existing library)
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFacePipeline
from dotenv import load_dotenv

load_dotenv()

warnings.filterwarnings("ignore")

c:\Users\lovep\miniconda3\envs\ragApp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# --- Setup ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # Use GPU if available, else fallback to CPU
DB_PATH = os.getenv("VECTOR_DB_PATH", "db/chroma")  # Vector DB storage path, overridable via .env

print(f"Using {'GPU' if DEVICE == 'cuda' else 'CPU'} for compute.")

Using GPU for compute.


In [ ]:
def partition_document(file_path: str) -> List[Document]:
    "Extract elements from PDF using unstructured."

    elements = partition_pdf(
        filename=file_path,  # Path to your PDF file
        strategy="fast", # Use the most accurate (but slower) processing method of extraction
        infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
        extract_image_block_types=["Image"], # Grab images found in the PDF (don't ignore images)
        extract_image_block_to_payload=True # Store images as base64 data you can actually use
    )

    print(f"Extracted {len(elements)} elements from {file_path}.")
    return elements

file_path = "NIPS-2017-attention-is-all-you-need-Paper.pdf"
elements = partition_document(file_path)

set([str(type(e)) for e in elements])  # Show the types of elements extracted

In [ ]:
elements[30].to_dict()  # Show the content of a specific element